## Question 2 — Scaled dot-product attention

$Q$ contains query vectors, $K$ matching key vectors, and $V$ content vectors. For $n$ self-attending tokens, $Q,K\in\mathbb R^{n\times d_k}$ and $V\in\mathbb R^{n\times d_v}$. Therefore $QK^T\in\mathbb R^{n\times n}$, weights are $n\times n$, and output is $n\times d_v$. If query/key components have roughly zero mean and unit variance, the variance of their sum-of-$d_k$ dot product grows roughly like $d_k$; division by $\sqrt{d_k}$ keeps scores nearer a useful softmax range.

**Imports.** NumPy computes the scores, the softmax, and the check.

In [8]:
import numpy as np

**Stable softmax.** Subtract the row max before the exponential so large scores do not overflow.

In [9]:
def stable_softmax(x, axis=-1):
    """Numerically stable softmax along one axis."""
    x = np.asarray(x, dtype=float)
    shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_shifted = np.exp(shifted)
    return exp_shifted / np.sum(exp_shifted, axis=axis, keepdims=True)

**Scaled dot-product attention.** Scores are $QK^\top/\sqrt{d_k}$. Softmax is applied to each query row, then those weights mix the values.

In [10]:
def scaled_dot_product_attention(Q, K, V):
    """Q:(nq, dk), K:(nk, dk), V:(nk, dv) -> output (nq, dv), weights (nq, nk)."""
    Q, K, V = (np.asarray(z, dtype=float) for z in (Q, K, V))
    if Q.ndim != 2 or K.ndim != 2 or V.ndim != 2 or Q.shape[1] != K.shape[1] or K.shape[0] != V.shape[0]:
        raise ValueError("Incompatible Q, K, V dimensions")
    scores = (Q @ K.T) / np.sqrt(Q.shape[1])
    weights = stable_softmax(scores, axis=-1)
    return weights @ V, weights

**Check on the 2 × 2 identity.** Each weight row sums to 1. Rounded to three decimals, the output is

$$
\begin{bmatrix} 0.670 & 0.330 \\ 0.330 & 0.670 \end{bmatrix}.
$$

In [11]:
identity = np.eye(2)
out2, weights2 = scaled_dot_product_attention(identity, identity, identity)
print("scores:\n", np.round(identity @ identity.T / np.sqrt(2), 6))
print("weights:\n", np.round(weights2, 6))
print("output:\n", np.round(out2, 6))
np.testing.assert_allclose(weights2.sum(axis=1), 1)
np.testing.assert_allclose(np.round(out2, 3), [[0.670, 0.330], [0.330, 0.670]])
print("Identity check passed.")

scores:
 [[0.707107 0.      ]
 [0.       0.707107]]
weights:
 [[0.669762 0.330238]
 [0.330238 0.669762]]
output:
 [[0.669762 0.330238]
 [0.330238 0.669762]]
Identity check passed.
